In [1]:
# workflow.py

from typing import Dict, Any
from langgraph.graph import StateGraph, END
from schemas import WorkflowState
from nodes import (
    extract_context_node,
    work_package_classification_node,   # New import
    qdrant_insertion_node,
    action_parameter_extraction_node,
    action_execution_node,
    conditional_routing,  # will be overridden below
)

# NODE REGISTRY
nodes = {
    "extract_context": extract_context_node,
    "classify_work_packages": work_package_classification_node, 
    "extract_params": action_parameter_extraction_node,
    "execute_actions": action_execution_node,
    "qdrant_insertion": qdrant_insertion_node,
    "conditional_router": conditional_routing,  # keep for future use
    "end": END,
}

# WORKFLOW 
def create_workflow_graph():
    workflow = StateGraph(WorkflowState)

    # Nodes
    workflow.add_node("extract_context", nodes["extract_context"])
    workflow.add_node("classify_work_packages", nodes["classify_work_packages"])
    workflow.add_node("extract_params", nodes["extract_params"])
    workflow.add_node("qdrant_insertion", nodes["qdrant_insertion"])
    workflow.add_node("execute_actions", nodes["execute_actions"])

    # Entry point
    workflow.set_entry_point("extract_context")

    # Routing with updated logic
    workflow.add_conditional_edges(
    "extract_context",
    conditional_routing,   # just use your updated function
    {
        "classify_work_packages": "classify_work_packages",  # route here
        "END": nodes["end"],                                # or end
    }
)

    # Linear flow
    workflow.add_edge("classify_work_packages", "extract_params")
    workflow.add_edge("extract_params", "qdrant_insertion")
    workflow.add_edge("qdrant_insertion", "execute_actions")
    workflow.add_edge("execute_actions", nodes["end"])

    return workflow.compile()


ImportError: cannot import name 'WorkflowState' from 'schemas' (unknown location)

In [2]:
pip install langgraph


Note: you may need to restart the kernel to use updated packages.


In [60]:
from pydantic import BaseModel
from typing import List, Dict, Any, Optional
from typing import TypedDict

class WhatsAppState(TypedDict):
    whatsapp_messages: List[Dict]
    sites_string: str
    extraction_result: Optional[SiteIdExtraction]
    # Extracted data for next nodes
    site_ids: Optional[List[str]]
    site_names: Optional[List[str]]
    actions: Optional[List[str]]
    chat_id: Optional[List[str]]
    From: Optional[str]
    confidence: Optional[float]
    reasoning: Optional[str]
    # Context for parameter extraction
    context: Optional[Dict[str, Any]]
    error: Optional[str]


In [61]:
messages=[
  {
    "chat_id": "919398348980@s.whatsapp.net",
    "chat_name": "Me",
    "chat_type": "individual",
    "from_user": "919398348980@s.whatsapp.net",
    "message": "Heyy",
    "context": [],
    "timestamp": "2025-09-24T07:05:33+05:30"
  },
  {
    "chat_id": "919398348980@s.whatsapp.net",
    "chat_name": "Me",
    "chat_type": "individual",
    "from_user": "919398348980@s.whatsapp.net",
    "message": "Heyy",
    "context": [
      {
        "from": "919398348980@s.whatsapp.net",
        "to": "919398348980@s.whatsapp.net",
        "chat_name": "Me",
        "chat_type": "individual",
        "message_id": "ACDEAFCF601159DB012A3D090A64E8A7",
        "timestamp": "2025-09-24T07:05:33+05:30",
        "message": "Heyy"
      }
    ],
    "timestamp": "2025-09-24T07:05:33+05:30"
  }
]


In [62]:
from typing import List, Dict, Any, Optional
from pydantic import BaseModel, Field

class SiteIdExtraction(BaseModel):
    site_ids: Optional[List[Optional[str]]] = Field(default=None)
    site_names: Optional[List[Optional[str]]] = Field(default=None)
    actions: List[str] = Field(...)
    chat_id: List[str] = Field(...)
    From: str = Field(...)
    confidence: float = Field(...)
    reasoning: str = Field(...)

In [63]:
from config import OPENAI_CONFIG

In [64]:
import os
from langchain_openai import AzureChatOpenAI
llm = AzureChatOpenAI(
        azure_deployment=OPENAI_CONFIG['azure_deployment'],
        api_version=OPENAI_CONFIG['api_version'],
        azure_endpoint=OPENAI_CONFIG['azure_endpoint'],
        api_key=OPENAI_CONFIG['api_key'],
        temperature=0,
        max_tokens=None
    )

In [65]:
from langchain import PromptTemplate

context_prompt_template = PromptTemplate.from_template("""
You are an intelligent agent for infrastructure project management.

**PROCESS EACH NEW MESSAGE INDIVIDUALLY and return ONE action per message.**

### NEW MESSAGES:
{messages}

### Available sites: {sites}

### INSTRUCTIONS:

1. Extract for each new message:
   - One action per message
   - thread_id (use chat_id from WhatsApp)
   - message_id (use WhatsApp message_id if present, else null)
   - site info only if explicitly mentioned

2. Action rules:
   - `add_risk`: problems, delays, incidents — only if site is valid
   - `update_task`: task progress — only if site is valid
   - `update_risk`: mitigation completion — only if site is valid
   - `update`: general updates if site not mentioned or invalid
   - `ask_clarification`: ambiguous project-related message
   - `irrelevant`: casual, greetings, non-project content

### OUTPUT JSON FORMAT:

```json
{{
  "actions": ["action1", "action2"],
  "chat_id": ["chat1", "chat2"],
  "site_ids": ["site1", null],
  "site_names": ["Site Name 1", null],
  "From": "primary_sender",
  "confidence": 0.9,
  "reasoning": "Brief explanation of decisions"
}}
Arrays must match the number of NEW messages.

Use null for missing site info or message_id.

From, confidence, reasoning are single values for the batch.
""")

In [66]:
from pydantic import BaseModel, Field
from langchain.output_parsers import PydanticOutputParser



In [67]:
from typing import List, Dict, Any, Optional
from pydantic import BaseModel, Field
from langchain.schema import HumanMessage


class SiteIdExtraction(BaseModel):
    site_ids: Optional[List[Optional[str]]] = Field(default=None)
    site_names: Optional[List[Optional[str]]] = Field(default=None)
    actions: List[str] = Field(...)
    chat_id: List[str] = Field(...)
    From: str = Field(...)
    confidence: float = Field(...)
    reasoning: str = Field(...)

parser = PydanticOutputParser(return_id=False, pydantic_object=SiteIdExtraction)


def extract_whatsapp_context(
    whatsapp_messages: List[Dict],
    sites_string: str,
    llm,
) -> SiteIdExtraction:
    """
    Simple function to extract actions, site info, and chat_id from WhatsApp messages
    using an LLM. Directly sends messages to LLM with context.
    """

    # Prepare prompt for LLM
    prompt = context_prompt_template.format(
        messages=whatsapp_messages,
        sites=sites_string,
        format_instructions=parser.get_format_instructions()  # if your parser has instructions, include here
    )

    # Call LLM
    response = llm.invoke([HumanMessage(content=prompt)])
    result = parser.parse(response.content)  # structured output parser


    return SiteIdExtraction(
        site_ids=result.site_ids,
        site_names=result.site_names,
        actions=result.actions,
        chat_id=result.chat_id,
        From=result.From,
        confidence=result.confidence,
        reasoning=result.reasoning
    )


In [69]:
def whatsapp_context_extraction_node(state: WhatsAppState, llm) -> WhatsAppState:
    """
    Node function to extract WhatsApp context and add results to state.
    """
    try:
            
        # Get data from state
        whatsapp_messages = state.get("whatsapp_messages", [])
        sites_string = state.get("sites_string", "")
        
        if not whatsapp_messages:
            return {
                **state,
                "error": "No WhatsApp messages provided",
                "extraction_result": None
            }
        
        # Extract context
        extraction_result = extract_whatsapp_context(
            whatsapp_messages=whatsapp_messages,
            sites_string=sites_string,
            llm=llm
        )
        
        # Create context for next nodes (parameter extraction)
        context = {
            "reasoning": extraction_result.reasoning,
            "site_ids": extraction_result.site_ids,
            "site_names": extraction_result.site_names,
            "chat_id": extraction_result.chat_id,
            "From": extraction_result.From,
            "new_messages": whatsapp_messages,  # For parameter extraction templates
        }
        
        # Update state with results - individual fields for easy access
        return {
            **state,
            "extraction_result": extraction_result,
            "site_ids": extraction_result.site_ids,
            "site_names": extraction_result.site_names, 
            "actions": extraction_result.actions,
            "chat_id": extraction_result.chat_id,
            "From": extraction_result.From,
            "confidence": extraction_result.confidence,
            "reasoning": extraction_result.reasoning,
            "context": context,  # Ready for parameter extraction
            "error": None
        }
        
    except Exception as e:
        return {
            **state,
            "error": f"Context extraction failed: {str(e)}",
            "extraction_result": None
        }

In [70]:
from typing import List, Dict, Any, Optional, TypedDict
from pydantic import BaseModel, Field
from langchain.schema import HumanMessage
from langchain.output_parsers import PydanticOutputParser
import logging
import json

logger = logging.getLogger(__name__)

# Same models as your email system
class Risk(BaseModel):
    site_id: str = Field(...)
    title: str = Field(...)
    description: str = Field(...)
    severity: str = Field(...)
    status: str = Field(default="open")
    category: str = Field(...)

class AddRiskInput(BaseModel):
    risks: List[Risk] = Field(...)
    title: str = Field(...)
    summary: str = Field(...)
    reasoning: str = Field(...)

class Task(BaseModel):
    task_id: str = Field(...)
    status: str = Field(...)
    notes: Optional[str] = Field(default=None)
    completion_percentage: Optional[int] = Field(default=None)

class UpdateTaskInput(BaseModel):
    tasks: List[Task] = Field(...)
    summary: str = Field(...)
    title: str = Field(...)
    reasoning: str = Field(...)

class RiskUpdate(BaseModel):
    risk_id: str = Field(...)
    status: str = Field(...)
    notes: Optional[str] = Field(default=None)
    severity: Optional[str] = Field(default=None)

class UpdateRiskInput(BaseModel):
    risks: List[RiskUpdate] = Field(...)
    summary: str = Field(...)
    title: str = Field(...)
    reasoning: str = Field(...)

# Parsers
risk_parser = PydanticOutputParser(return_id=False, pydantic_object=AddRiskInput)
task_parser = PydanticOutputParser(return_id=False, pydantic_object=UpdateTaskInput)
risk_update_parser = PydanticOutputParser(return_id=False, pydantic_object=UpdateRiskInput)

# Updated State definition
class WhatsAppState(TypedDict):
    whatsapp_messages: List[Dict]
    sites_string: str
    extraction_result: Optional[SiteIdExtraction]
    # Extracted data for next nodes
    site_ids: Optional[List[str]]
    site_names: Optional[List[str]]
    actions: Optional[List[str]]
    chat_id: Optional[List[str]]
    From: Optional[str]
    confidence: Optional[float]
    reasoning: Optional[str]
    context: Optional[Dict[str, Any]]
    # Action parameter results
    action_parameters: Optional[Dict[str, BaseModel]]
    parameter_extraction_results: Optional[List[Dict[str, Any]]]
    error: Optional[str]

# Templates (same as your email system)
risk_prompt_template = """
Based on the following WhatsApp conversation, extract risk information to add:

WhatsApp Messages: {new_messages}
Available Sites: {available_sites}
Site Names: {site_names}
From: {From}
Available Tasks: {available_tasks}
Available Risks: {available_risks}
Reasoning: {reasoning}

{format_instructions}
"""

task_prompt_template = """
Based on the following WhatsApp conversation, extract task updates:

WhatsApp Messages: {new_messages}
Available Tasks: {available_tasks}
Reasoning: {reasoning}
From: {From}

{format_instructions}
"""

risk_update_prompt_template = """
Based on the following WhatsApp conversation, extract risk updates:

WhatsApp Messages: {new_messages}
Available Tasks with Risks: {available_tasks_with_risks}
Reasoning: {reasoning}
From: {From}

{format_instructions}
"""

def get_risks_by_site(db, site_id):
    """Mock function - replace with your actual DB query"""
    # Return mock risks for now
    return [
        {
            "_id": "risk_123",
            "title": "Mock Risk",
            "site_id": site_id
        }
    ]

def execute_whatsapp_action_parameters(
    action: str,
    context: Dict[str, Any],
    site_ids: Optional[List[str]] = None,
    site_names: Optional[List[str]] = None,
    From: str = "",
    tasks_for_site: Optional[Dict] = None,
    packages_for_site: Optional[Dict] = None,
    llm=None,
    db=None,
) -> BaseModel:
    """Use LLM to extract parameters but return structured Pydantic objects.

    The signature is kept backward-compatible with WhatsApp callers.
    """
    

    reasoning = context.get("reasoning", "")
    new_messages_content = context.get("new_messages", "")
    
    # Prefer the values passed in, but fall back to those present in context
    site_ids = site_ids or context.get("site_ids", [])
    site_names = site_names or context.get("site_names", [])
    iwps = context.get("iwps", {})
    
    # Ensure availability maps are always dicts to avoid KeyErrors
    tasks_for_site = (tasks_for_site or context.get("tasks_for_site") or {})
    packages_for_site = (packages_for_site or context.get("packages_for_site") or {})

    # Aggregate risks for all available site_ids to provide context to LLM
    risks: List[Dict[str, Any]] = []
    try:
        if isinstance(site_ids, list):
            for sid in site_ids:
                if sid:  # Check if site_id is not None
                    risks.extend(get_risks_by_site(db, sid))
        elif isinstance(site_ids, str) and site_ids:
            risks = get_risks_by_site(db, site_ids)
        else:
            risks = []
    except Exception as e:
        logger.error(f"Error aggregating risks for sites {site_ids}: {e}")
        risks = []
    
    # Visibility into what the LLM will see for update_risk grounding
    try:
        if isinstance(site_ids, list):
            logger.info("update_risk grounding: site_ids=%s open_risks_found=%d", site_ids, len(risks))
        else:
            logger.info("update_risk grounding: site_id=%s open_risks_found=%d", site_ids, len(risks))
    except Exception:
        pass
    
    if action == "add_risk":
        try:
            # Extract primary site values instead of passing lists
            primary_site_id = site_ids[0] if site_ids and len(site_ids) > 0 else None
            primary_site_name = site_names[0] if site_names and len(site_names) > 0 else None
            
            logger.debug(f"add_risk extraction: site_ids={site_ids}, site_names={site_names}")
            logger.debug(f"add_risk extraction: primary_site_id={primary_site_id}, primary_site_name={primary_site_name}")
            
            # Provide available_tasks to the template to avoid KeyError
            risk_prompt = risk_prompt_template.format(
                new_messages=json.dumps(new_messages_content, indent=2),
                available_sites=primary_site_id,
                site_names=primary_site_name,
                From=From,
                available_tasks=json.dumps(tasks_for_site, indent=2),
                available_risks=json.dumps(risks, indent=2),
                reasoning=reasoning,
                format_instructions=risk_parser.get_format_instructions()
            )
            
            logger.debug(f"Risk prompt created with site_id: {primary_site_id}")
            
            risk_response = llm.invoke([HumanMessage(content=risk_prompt)])
            parsed_risk_response = risk_parser.parse(risk_response.content)
            
            logger.debug(f"DEBUG - Parsed risk response: {parsed_risk_response}")
            
            # Ensure each risk has correct site_id and validate data types
            for i, risk in enumerate(parsed_risk_response.risks):
                logger.debug(f"Risk {i}: site_id={risk.site_id} (type: {type(risk.site_id)})")
                
                # Fix site_id if it's None or ensure it's the primary site
                if not risk.site_id and primary_site_id:
                    risk.site_id = primary_site_id
                    logger.debug(f"Set missing site_id to: {risk.site_id}")
                elif isinstance(risk.site_id, list):
                    # This shouldn't happen now, but just in case
                    risk.site_id = risk.site_id[0] if risk.site_id else primary_site_id
                    logger.warning(f"Fixed list site_id to: {risk.site_id}")
                
                # Ensure it's a string
                if risk.site_id:
                    risk.site_id = str(risk.site_id)
            
            logger.debug(f"Final risk site_ids: {[risk.site_id for risk in parsed_risk_response.risks]}")
            return parsed_risk_response
            
        except Exception as e:
            logger.error(f"Error in add_risk parameter extraction: {str(e)}")
            return AddRiskInput(
                risks=[],
                title="",
                summary=f"Risk parameter extraction failed: {str(e)}"
            )
        
    elif action == "update_task":
        try:
            task_prompt = task_prompt_template.format(
                new_messages=json.dumps(new_messages_content, indent=2),
                available_tasks=json.dumps(iwps, indent=2),
                reasoning=reasoning,
                From=From,
                format_instructions=task_parser.get_format_instructions()
            )

            task_response = llm.invoke([HumanMessage(content=task_prompt)])
            parsed_task_response = task_parser.parse(task_response.content)

            logger.debug(f"Parsed task response: {parsed_task_response.tasks}")
            return UpdateTaskInput(
                tasks=parsed_task_response.tasks,
                summary=parsed_task_response.summary,
                title=parsed_task_response.title,
                reasoning=getattr(parsed_task_response, 'reasoning', 'Task status updated based on WhatsApp communication')
            )
        except Exception as e:
            logger.error(f"Error in update_task parameter extraction: {str(e)}")
            return UpdateTaskInput(
                tasks=[],
                title="",
                summary=f"Task parameter extraction failed: {str(e)}",
                reasoning=str(e)
            )
        
    elif action == "update_risk":
        try:
            logger.debug("Using LLM to update the risk")
            
            # Provide concise open-risk list with explicit risk_id to help the LLM
            try:
                concise_risks = []
                for r in risks or []:
                    try:
                        concise_risks.append({
                            "risk_id": str(r.get("_id", "")),
                            "title": r.get("title", ""),
                            "site_id": str(r.get("site_id", ""))
                        })
                    except Exception:
                        continue
                available_tasks_with_risks = json.dumps(concise_risks, indent=2)
            except Exception:
                available_tasks_with_risks = "[]"
                
            risk_update_prompt = risk_update_prompt_template.format(
                new_messages=json.dumps(new_messages_content, indent=2),
                available_tasks_with_risks=available_tasks_with_risks,
                reasoning=reasoning,
                From=From,
                format_instructions=risk_update_parser.get_format_instructions()
            )
            
            risk_update_response = llm.invoke([HumanMessage(content=risk_update_prompt)])
            parsed_risk_update_response = risk_update_parser.parse(risk_update_response.content)
            
            logger.debug(f"Parsed risk update response: {parsed_risk_update_response}")
            return parsed_risk_update_response
            
        except Exception as e:
            logger.error(f"Error in update_risk parameter extraction: {str(e)}")
            return UpdateRiskInput(
                risks=[],
                title="",
                summary=f"Risk update parameter extraction failed: {str(e)}",
                reasoning=str(e)
            )
    
    else:
        logger.warning(f"Unknown action: {action}")
        return AddRiskInput(
            risks=[],
            title=f"Unknown Action: {action}",
            summary=f"Action '{action}' is not supported",
            reasoning=f"Unsupported action: {action}"
        )

def whatsapp_action_parameter_execution_node(state: WhatsAppState, llm, db=None) -> WhatsAppState:
    """
    Node function to execute action parameter extraction for all actions.
    """
    try:
        logger.info("Starting WhatsApp action parameter execution node")
        
        # Get data from state
        actions = state.get("actions", [])
        context = state.get("context", {})
        site_ids = state.get("site_ids", [])
        site_names = state.get("site_names", [])
        From = state.get("From", "")
        
        if not actions:
            logger.warning("No actions found in state")
            return {
                **state,
                "error": "No actions to execute",
                "action_parameters": {},
                "parameter_extraction_results": []
            }
        
        if not context:
            logger.warning("No context found in state")
            return {
                **state,
                "error": "No context available for parameter extraction",
                "action_parameters": {},
                "parameter_extraction_results": []
            }
        
        action_parameters = {}
        parameter_extraction_results = []
        
        # Execute parameter extraction for each action
        for action in actions:
            try:
                logger.info(f"Executing parameter extraction for action: {action}")
                
                # Normalize action name
                normalized_action = action.lower().replace(" ", "_")
                
                # Execute parameter extraction
                result = execute_whatsapp_action_parameters(
                    action=normalized_action,
                    context=context,
                    site_ids=site_ids,
                    site_names=site_names,
                    From=From,
                    tasks_for_site=context.get("tasks_for_site", {}),
                    packages_for_site=context.get("packages_for_site", {}),
                    llm=llm,
                    db=db
                )
                
                # Store results
                action_parameters[normalized_action] = result
                parameter_extraction_results.append({
                    "action": action,
                    "normalized_action": normalized_action,
                    "status": "success",
                    "parameters": result,
                    "message": f"Parameters extracted successfully for {action}"
                })
                
                logger.info(f"Parameter extraction successful for {action}")
                
            except Exception as e:
                logger.error(f"Error executing parameter extraction for action '{action}': {str(e)}")
                parameter_extraction_results.append({
                    "action": action,
                    "normalized_action": action.lower().replace(" ", "_"),
                    "status": "failed",
                    "error": str(e),
                    "message": f"Parameter extraction failed for {action}"
                })
        
        # Update state with results
        successful_extractions = len([r for r in parameter_extraction_results if r["status"] == "success"])
        logger.info(f"Parameter extraction completed: {successful_extractions}/{len(actions)} successful")
        
        return {
            **state,
            "action_parameters": action_parameters,
            "parameter_extraction_results": parameter_extraction_results,
            "error": None
        }
        
    except Exception as e:
        logger.error(f"Error in action parameter execution node: {str(e)}", exc_info=True)
        return {
            **state,
            "error": f"Action parameter execution failed: {str(e)}",
            "action_parameters": {},
            "parameter_extraction_results": []
        }

# Factory function for dependency injection
def create_whatsapp_action_parameter_node(llm, db=None):
    """
    Create action parameter execution node with dependencies injected.
    """
    def node(state: WhatsAppState) -> WhatsAppState:
        return whatsapp_action_parameter_execution_node(state, llm, db)
    
    return node

In [52]:
sites_string = "Site A, Site B, Site C"  # list of available sites

result = extract_whatsapp_context(messages, sites_string, llm)

# Print structured output
print(result.model_dump_json(indent=2))

{
  "site_ids": [
    null,
    null
  ],
  "site_names": [
    null,
    null
  ],
  "actions": [
    "irrelevant",
    "irrelevant"
  ],
  "chat_id": [
    "919398348980@s.whatsapp.net",
    "919398348980@s.whatsapp.net"
  ],
  "From": "919398348980@s.whatsapp.net",
  "confidence": 0.95,
  "reasoning": "Both messages contain only the casual greeting 'Heyy' with no project-related content or site mention, so they are classified as irrelevant."
}


In [71]:
from typing import List, Dict, Any, Optional, TypedDict
from pydantic import BaseModel, Field
import logging

logger = logging.getLogger(__name__)

# 1. ENHANCED STATE - to carry work package data between nodes
class WhatsAppState(TypedDict):
    # Input data
    whatsapp_messages: List[Dict]
    sites_string: str
    
    # Context extraction results
    extraction_result: Optional['SiteIdExtraction']
    site_ids: Optional[List[str]]
    site_names: Optional[List[str]]
    actions: Optional[List[str]]
    chat_id: Optional[List[str]]
    From: Optional[str]
    confidence: Optional[float]
    reasoning: Optional[str]
    context: Optional[Dict[str, Any]]
    
    # Work package classification results (NEW)
    tasks_for_site: Optional[Dict[str, List[Dict]]]  # site_id -> tasks
    packages_for_site: Optional[Dict[str, List[Dict]]]  # site_id -> packages
    iwps: Optional[Dict[str, Any]]  # integrated work packages
    risks_for_site: Optional[Dict[str, List[Dict]]]  # site_id -> risks
    
    # Action parameter results
    action_parameters: Optional[Dict[str, BaseModel]]
    parameter_extraction_results: Optional[List[Dict[str, Any]]]
    
    # Action execution results (NEW)
    execution_results: Optional[List[Dict[str, Any]]]
    executed_actions: Optional[Dict[str, Any]]
    
    # Error handling
    error: Optional[str]

# 2. REAL DATABASE FUNCTIONS - replace mock functions with actual DB queries

def get_risks_by_site(db, site_id: str) -> List[Dict[str, Any]]:
    """
    Fetch risks for a specific site from database.
    Replace with your actual database query.
    """
    try:
        if not db or not site_id:
            return []
        
        # Example MongoDB query - adjust for your database
        risks_collection = db.get_collection("risks")  # or however you access collections
        risks = list(risks_collection.find({
            "site_id": site_id,
            "status": {"$ne": "closed"}  # Only open risks
        }))
        
        # Convert ObjectId to string for JSON serialization
        for risk in risks:
            if "_id" in risk:
                risk["_id"] = str(risk["_id"])
        
        logger.info(f"Found {len(risks)} open risks for site {site_id}")
        return risks
        
    except Exception as e:
        logger.error(f"Error fetching risks for site {site_id}: {e}")
        return []

def get_tasks_for_site(db, site_id: str) -> List[Dict[str, Any]]:
    """
    Fetch tasks/work packages for a specific site from database.
    """
    try:
        if not db or not site_id:
            return []
        
        # Example query - adjust for your schema
        tasks_collection = db.get_collection("tasks")  # or "work_packages"
        tasks = list(tasks_collection.find({
            "site_id": site_id,
            "status": {"$nin": ["completed", "cancelled"]}  # Active tasks only
        }))
        
        # Convert ObjectId to string
        for task in tasks:
            if "_id" in task:
                task["_id"] = str(task["_id"])
        
        logger.info(f"Found {len(tasks)} active tasks for site {site_id}")
        return tasks
        
    except Exception as e:
        logger.error(f"Error fetching tasks for site {site_id}: {e}")
        return []

def get_packages_for_site(db, site_id: str) -> List[Dict[str, Any]]:
    """
    Fetch work packages for a specific site from database.
    """
    try:
        if not db or not site_id:
            return []
        
        # Example query - adjust for your schema
        packages_collection = db.get_collection("work_packages")
        packages = list(packages_collection.find({
            "site_id": site_id,
            "status": "active"
        }))
        
        # Convert ObjectId to string
        for package in packages:
            if "_id" in package:
                package["_id"] = str(package["_id"])
        
        logger.info(f"Found {len(packages)} work packages for site {site_id}")
        return packages
        
    except Exception as e:
        logger.error(f"Error fetching packages for site {site_id}: {e}")
        return []

# 3. WORK PACKAGE CLASSIFICATION NODE - to fetch tasks/packages for sites

def whatsapp_work_package_classification_node(state: WhatsAppState, db) -> WhatsAppState:
    """
    Node to classify and fetch work packages, tasks, and risks for extracted sites.
    """
    try:
        logger.info("Starting work package classification node")
        
        site_ids = state.get("site_ids", [])
        
        if not site_ids:
            logger.warning("No site IDs found for work package classification")
            return {
                **state,
                "tasks_for_site": {},
                "packages_for_site": {},
                "risks_for_site": {},
                "iwps": {}
            }
        
        if not db:
            logger.error("Database connection not available")
            return {
                **state,
                "error": "Database connection required for work package classification",
                "tasks_for_site": {},
                "packages_for_site": {},
                "risks_for_site": {},
                "iwps": {}
            }
        
        tasks_for_site = {}
        packages_for_site = {}
        risks_for_site = {}
        iwps = {}
        
        # Fetch data for each site
        for site_id in site_ids:
            if not site_id:  # Skip None values
                continue
                
            logger.info(f"Fetching work packages for site: {site_id}")
            
            # Fetch tasks
            site_tasks = get_tasks_for_site(db, site_id)
            tasks_for_site[site_id] = site_tasks
            
            # Fetch packages
            site_packages = get_packages_for_site(db, site_id)
            packages_for_site[site_id] = site_packages
            
            # Fetch risks
            site_risks = get_risks_by_site(db, site_id)
            risks_for_site[site_id] = site_risks
            
            # Create integrated work packages (iwps) structure
            iwps[site_id] = {
                "tasks": site_tasks,
                "packages": site_packages,
                "risks": site_risks,
                "site_id": site_id
            }
        
        # Update context with fetched data
        existing_context = state.get("context", {})
        updated_context = {
            **existing_context,
            "tasks_for_site": tasks_for_site,
            "packages_for_site": packages_for_site,
            "risks_for_site": risks_for_site,
            "iwps": iwps
        }
        
        logger.info(f"Work package classification completed for {len(site_ids)} sites")
        
        return {
            **state,
            "tasks_for_site": tasks_for_site,
            "packages_for_site": packages_for_site,
            "risks_for_site": risks_for_site,
            "iwps": iwps,
            "context": updated_context,
            "error": None
        }
        
    except Exception as e:
        logger.error(f"Error in work package classification: {str(e)}", exc_info=True)
        return {
            **state,
            "error": f"Work package classification failed: {str(e)}",
            "tasks_for_site": {},
            "packages_for_site": {},
            "risks_for_site": {},
            "iwps": {}
        }

def create_work_package_classification_node(db):
    """
    Factory function for work package classification node.
    """
    def node(state: WhatsAppState) -> WhatsAppState:
        return whatsapp_work_package_classification_node(state, db)
    return node


In [72]:
from typing import List, Dict, Any, Optional, TypedDict
from pydantic import BaseModel, Field
import logging

logger = logging.getLogger(__name__)

# Import your existing database functions directly
from your_email_actions import (
    add_risk,
    update_task, 
    update_risk,
    update_response_in_db,
    create_communication_log,
    get_risks_by_site,
    get_tasks_for_site,
    get_packages_for_site
)

# Enhanced State - to carry work package data between nodes
class WhatsAppState(TypedDict):
    # Input data
    whatsapp_messages: List[Dict]
    sites_string: str
    
    # Context extraction results
    extraction_result: Optional['SiteIdExtraction']
    site_ids: Optional[List[str]]
    site_names: Optional[List[str]]
    actions: Optional[List[str]]
    chat_id: Optional[List[str]]
    From: Optional[str]
    confidence: Optional[float]
    reasoning: Optional[str]
    context: Optional[Dict[str, Any]]
    
    # Work package classification results (NEW)
    tasks_for_site: Optional[Dict[str, List[Dict]]]  # site_id -> tasks
    packages_for_site: Optional[Dict[str, List[Dict]]]  # site_id -> packages
    iwps: Optional[Dict[str, Any]]  # integrated work packages
    risks_for_site: Optional[Dict[str, List[Dict]]]  # site_id -> risks
    
    # Action parameter results
    action_parameters: Optional[Dict[str, BaseModel]]
    parameter_extraction_results: Optional[List[Dict[str, Any]]]
    
    # Action execution results (NEW)
    execution_results: Optional[List[Dict[str, Any]]]
    executed_actions: Optional[Dict[str, Any]]
    
    # Error handling
    error: Optional[str]

# Work Package Classification Node - fetches tasks/packages for sites
def whatsapp_work_package_classification_node(state: WhatsAppState, db) -> WhatsAppState:
    """
    Node to classify and fetch work packages, tasks, and risks for extracted sites.
    Uses your existing database functions.
    """
    try:
        logger.info("Starting WhatsApp work package classification node")
        
        site_ids = state.get("site_ids", [])
        
        if not site_ids:
            logger.warning("No site IDs found for work package classification")
            return {
                **state,
                "tasks_for_site": {},
                "packages_for_site": {},
                "risks_for_site": {},
                "iwps": {}
            }
        
        if not db:
            logger.error("Database connection not available")
            return {
                **state,
                "error": "Database connection required for work package classification",
                "tasks_for_site": {},
                "packages_for_site": {},
                "risks_for_site": {},
                "iwps": {}
            }
        
        tasks_for_site = {}
        packages_for_site = {}
        risks_for_site = {}
        iwps = {}
        
        # Fetch data for each site using your existing functions
        for site_id in site_ids:
            if not site_id:
                continue
                
            logger.info(f"Fetching work packages for site: {site_id}")
            
            # Use your existing database functions
            site_risks = get_risks_by_site(db, site_id)
            risks_for_site[site_id] = site_risks
            
            # Get tasks if you have these functions, otherwise empty
            try:
                site_tasks = get_tasks_for_site(db, site_id)
                tasks_for_site[site_id] = site_tasks
            except:
                tasks_for_site[site_id] = []
            
            try:
                site_packages = get_packages_for_site(db, site_id)
                packages_for_site[site_id] = site_packages
            except:
                packages_for_site[site_id] = []
            
            # Create iwps structure
            iwps[site_id] = {
                "tasks": tasks_for_site[site_id],
                "packages": packages_for_site[site_id], 
                "risks": site_risks,
                "site_id": site_id
            }
        
        # Update context with fetched data
        existing_context = state.get("context", {})
        updated_context = {
            **existing_context,
            "tasks_for_site": tasks_for_site,
            "packages_for_site": packages_for_site,
            "risks_for_site": risks_for_site,
            "iwps": iwps
        }
        
        logger.info(f"Work package classification completed for {len(site_ids)} sites")
        
        return {
            **state,
            "tasks_for_site": tasks_for_site,
            "packages_for_site": packages_for_site,
            "risks_for_site": risks_for_site,
            "iwps": iwps,
            "context": updated_context,
            "error": None
        }
        
    except Exception as e:
        logger.error(f"Error in work package classification: {str(e)}", exc_info=True)
        return {
            **state,
            "error": f"Work package classification failed: {str(e)}",
            "tasks_for_site": {},
            "packages_for_site": {},
            "risks_for_site": {},
            "iwps": {}
        }

def create_work_package_classification_node(db):
    """Factory function for work package classification node."""
    def node(state: WhatsAppState) -> WhatsAppState:
        return whatsapp_work_package_classification_node(state, db)
    return node

# Action Execution Node - uses your existing action functions
def whatsapp_action_execution_node(state: WhatsAppState, db) -> WhatsAppState:
    """
    Node to execute the actual actions using your existing email system functions.
    """
    try:
        logger.info("Starting WhatsApp action execution node")
        
        action_parameters = state.get("action_parameters", {})
        From = state.get("From", "whatsapp_user")
        
        if not action_parameters:
            logger.warning("No action parameters found for execution")
            return {
                **state,
                "execution_results": [],
                "executed_actions": {},
                "error": "No actions to execute"
            }
        
        if not db:
            logger.error("Database connection not available for action execution")
            return {
                **state,
                "error": "Database connection required for action execution",
                "execution_results": [],
                "executed_actions": {}
            }
        
        execution_results = []
        executed_actions = {}
        
        # Execute each action using your existing functions
        for action_name, parameters in action_parameters.items():
            try:
                logger.info(f"Executing WhatsApp action: {action_name}")
                
                if action_name == "add_risk":
                    # Convert Pydantic models to dicts for your existing function
                    risks_data = []
                    for risk in parameters.risks:
                        risk_dict = {
                            "site_id": risk.site_id,
                            "description": risk.description,
                            "severity": risk.severity,
                            "impact": getattr(risk, 'impact', 'medium'),
                            "mitigation_plan": getattr(risk, 'mitigation_plan', ''),
                            "reasoning": getattr(risk, 'reasoning', ''),
                            "package_id": getattr(risk, 'package_id', None)
                        }
                        risks_data.append(risk_dict)
                    
                    # Use your existing add_risk function
                    result = add_risk(
                        risks=risks_data,
                        summary=parameters.summary,
                        title=parameters.title,
                        From=From,
                        db=db,
                        work_package_data=None  # Add work package data if needed
                    )
                    
                    executed_actions[action_name] = {
                        "risks_added": len(parameters.risks),
                        "result": result
                    }
                    
                    execution_results.append({
                        "action": action_name,
                        "status": "success",
                        "message": f"Added {len(parameters.risks)} risks via WhatsApp",
                        "details": result
                    })
                
                elif action_name == "update_task":
                    # Convert Pydantic models to dicts
                    tasks_data = []
                    for task in parameters.tasks:
                        task_dict = {
                            "task_id": task.task_id,
                            "status": task.status,
                            "notes": task.notes,
                            "completion_percentage": task.completion_percentage,
                            "reasoning": getattr(task, 'reasoning', '')
                        }
                        tasks_data.append(task_dict)
                    
                    # Use your existing update_task function
                    result = update_task(
                        tasks=tasks_data,
                        summary=parameters.summary,
                        title=parameters.title,
                        From=From,
                        db=db,
                        work_package_data=None
                    )
                    
                    executed_actions[action_name] = {
                        "tasks_updated": len(parameters.tasks),
                        "result": result
                    }
                    
                    execution_results.append({
                        "action": action_name,
                        "status": "success",
                        "message": f"Updated {len(parameters.tasks)} tasks via WhatsApp",
                        "details": result
                    })
                
                elif action_name == "update_risk":
                    # Convert Pydantic models to dicts
                    risks_data = []
                    for risk_update in parameters.risks:
                        risk_dict = {
                            "risk_id": risk_update.risk_id,
                            "status": risk_update.status,
                            "notes": risk_update.notes,
                            "severity": risk_update.severity,
                            "mitigation_plan": getattr(risk_update, 'mitigation_plan', ''),
                            "reasoning": getattr(risk_update, 'reasoning', ''),
                            "package_id": getattr(risk_update, 'package_id', None)
                        }
                        risks_data.append(risk_dict)
                    
                    # Use your existing update_risk function
                    result = update_risk(
                        risks=risks_data,
                        summary=parameters.summary,
                        title=parameters.title,
                        From=From,
                        db=db,
                        work_package_data=None
                    )
                    
                    executed_actions[action_name] = {
                        "risks_updated": len(parameters.risks),
                        "result": result
                    }
                    
                    execution_results.append({
                        "action": action_name,
                        "status": "success", 
                        "message": f"Updated {len(parameters.risks)} risks via WhatsApp",
                        "details": result
                    })
                
                else:
                    # Handle unknown actions by creating a communication log
                    result = create_communication_log(
                        communication_type="whatsapp_message",
                        message=f"Received WhatsApp message with unknown action: {action_name}",
                        title=f"Unknown Action: {action_name}",
                        status="open",
                        From=From,
                        db=db,
                        work_package_data=None
                    )
                    
                    execution_results.append({
                        "action": action_name,
                        "status": "logged",
                        "message": f"Unknown action logged as communication",
                        "details": result
                    })
                
            except Exception as e:
                logger.error(f"Error executing WhatsApp action {action_name}: {str(e)}")
                execution_results.append({
                    "action": action_name,
                    "status": "failed",
                    "message": f"Execution failed: {str(e)}"
                })
        
        successful_actions = len([r for r in execution_results if r["status"] in ["success", "logged"]])
        logger.info(f"WhatsApp action execution completed: {successful_actions}/{len(action_parameters)} successful")
        
        return {
            **state,
            "execution_results": execution_results,
            "executed_actions": executed_actions,
            "error": None
        }
        
    except Exception as e:
        logger.error(f"Error in WhatsApp action execution node: {str(e)}", exc_info=True)
        return {
            **state,
            "error": f"Action execution failed: {str(e)}",
            "execution_results": [],
            "executed_actions": {}
        }

def create_action_execution_node(db):
    """Factory function for action execution node."""
    def node(state: WhatsAppState) -> WhatsAppState:
        return whatsapp_action_execution_node(state, db)
    return node

# Updated parameter extraction to use your existing get_risks_by_site function
def execute_whatsapp_action_parameters(
    action: str,
    context: Dict[str, Any],
    site_ids: Optional[List[str]] = None,
    site_names: Optional[List[str]] = None,
    From: str = "",
    tasks_for_site: Optional[Dict] = None,
    packages_for_site: Optional[Dict] = None,
    llm=None,
    db=None,
) -> BaseModel:
    """Use LLM to extract parameters but return structured Pydantic objects.
    Now uses your existing get_risks_by_site function."""
    
    reasoning = context.get("reasoning", "")
    new_messages_content = context.get("new_messages", "")
    
    # Prefer the values passed in, but fall back to those present in context
    site_ids = site_ids or context.get("site_ids", [])
    site_names = site_names or context.get("site_names", [])
    iwps = context.get("iwps", {})
    
    # Ensure availability maps are always dicts to avoid KeyErrors
    tasks_for_site = (tasks_for_site or context.get("tasks_for_site") or {})
    packages_for_site = (packages_for_site or context.get("packages_for_site") or {})

    # Use your existing get_risks_by_site function
    risks: List[Dict[str, Any]] = []
    try:
        if isinstance(site_ids, list):
            for sid in site_ids:
                if sid:  # Check if site_id is not None
                    risks.extend(get_risks_by_site(db, sid))
        elif isinstance(site_ids, str) and site_ids:
            risks = get_risks_by_site(db, site_ids)
        else:
            risks = []
    except Exception as e:
        logger.error(f"Error aggregating risks for sites {site_ids}: {e}")
        risks = []
    
    # Rest of your parameter extraction logic remains the same...
    # (The add_risk, update_task, update_risk logic you already have)
    
    # Import your existing parameter extraction models and logic
    from your_existing_models import (
        AddRiskInput, UpdateTaskInput, UpdateRiskInput,
        risk_parser, task_parser, risk_update_parser,
        risk_prompt_template, task_prompt_template, risk_update_prompt_template
    )
    
    # Your existing parameter extraction logic goes here...
    # Just replace the mock functions with your real ones

ModuleNotFoundError: No module named 'your_email_actions'

In [1]:
from agent.startup import initialize_dependencies
db, llm, embedding_model, qdrant_client = initialize_dependencies()

In [2]:
db.name

'alfreddemo'